# 02-chunk · 03 — store backends, selected by a flag

**Only one of the three backends below actually runs in this cookbook — Chroma, because it's file-based and needs no server or account; Pinecone and S3 Vectors are real, wired code that stop at a named error the moment a credential is missing.**

One chunker, three possible destinations (Pinecone, S3 Vectors, local
ChromaDB) selected by CLI flags (`--pinecone-only`, `--s3-only`,
`--chroma`/`--chroma-only`), plus resume/dry-run bookkeeping
(`progress_path()`, `load_progress()`, `save_progress()` — one progress file
per destination, so a paper written to one index is not wrongly treated as
"done" when the target changes).

This repo ships with no cloud credentials, so Pinecone and S3 Vectors below
are wired to the same interface as Chroma but stop at a clearly-labeled stub
the moment they would need a real key — `get_pinecone_index_stub()` raises a
named `"Set PINECONE_API_KEY"` error rather than failing deep inside a
client library, surfaced through the shared `upsert()` dispatcher. The
resume check is built and proven working on its own (Step 6) before it is
ever wired into a real store write (Step 11).

## What this notebook demonstrates

| Name | What it does | Example |
| --- | --- | --- |
| `chunk_document_to_records(doc, doc_id, doc_metadata)` | Condensed reuse of notebook 02's `HybridChunker` pipeline | 2 chunks for the outcomes paper, 1 for the discussion paper |
| `progress_path(target)` | One progress-file path per destination name | `progress_path("synthetic_papers")` -> a temp-dir path |
| `load_progress(path)` / `save_progress(path, ids)` | Read/write the set of doc ids already written to a given destination | empty set before any write, populated after |
| `hash_embed(texts)` | Deterministic offline embedding stand-in, same as notebook 01 | 16-dim vectors, no API key needed |
| `get_chroma_collection(db_dir, name)` | Opens (or creates) a local, file-based Chroma collection | a cosine-space collection, 0 vectors before any upsert |
| `upsert_chroma(col, records, embeddings)` | Writes chunk records + vectors into a Chroma collection | collection count goes from 0 to N |
| `get_pinecone_index_stub(index_name)` | Raises a named error if `PINECONE_API_KEY` isn't set, otherwise returns a real Pinecone index | `ValueError("Set PINECONE_API_KEY...")` with no key |
| `get_s3vectors_stub()` | Raises a named error if AWS credentials aren't set | `ValueError(...)` with no key |
| `upsert(backend, records, embeddings, **kwargs)` | One dispatcher over all three backends, keyed by a flag | `upsert("chroma", records, embeddings, ...)` |
| `run(backend, docs, dry_run, no_resume, **kwargs)` | Ties chunking + resume bookkeeping + the store dispatcher into one flag-selected entry point | dry-run, real run, resumed run, gated Pinecone run |


In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()


## Step 1 — sqlite3 patch (older-Linux gotcha)

`chromadb` requires sqlite3 >= 3.35.0; some Linux clusters ship an older
system sqlite3. The fix — swap in `pysqlite3-binary` before `chromadb` is
imported — is guarded so it's a no-op on a system where the system sqlite3
is already new enough.

In [ ]:
import sys

try:
    __import__("pysqlite3")
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass

import chromadb
print("chromadb", chromadb.__version__, "- sqlite3", __import__("sqlite3").sqlite_version)


## Step 2 — the chunker (condensed reuse of notebook 02)

Per this repo's rule that every notebook carries real, runnable code rather
than importing a shared module, the `HybridChunker` path is repeated here in
condensed form — same logic as notebook `02`, so see that notebook for the
line-by-line provenance and the version-drift note on `doc_items` resolution.
This is a deliberate exception to "one function per cell": notebook `02`
already teaches each of these pieces separately, so this notebook treats them
as a single given building block, not new material.

In [ ]:
import tempfile
from pathlib import Path
from typing import Any
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

MIN_CHUNK_WORDS = 25


def _item_label(item: Any) -> str:
    label = getattr(item, "label", None)
    return str(getattr(label, "value", label) or "").lower()


def _resolve_doc_item(item: Any, doc: Any) -> Any:
    ref = getattr(item, "self_ref", "") or ""
    try:
        if ref.startswith("#/tables/"):
            return doc.tables[int(ref.rsplit("/", 1)[-1])]
        if ref.startswith("#/pictures/"):
            return doc.pictures[int(ref.rsplit("/", 1)[-1])]
    except (IndexError, ValueError):
        pass
    return item


def _try_table_markdown(doc_items: list[Any], doc: Any) -> str:
    for raw in doc_items:
        item = _resolve_doc_item(raw, doc)
        if _item_label(item) == "table" and hasattr(item, "export_to_markdown"):
            try:
                return item.export_to_markdown(doc=doc)
            except Exception:
                pass
    return ""


def get_chunk_type(chunk_obj: Any, doc: Any) -> str:
    doc_items = list(getattr(getattr(chunk_obj, "meta", None), "doc_items", []) or [])
    labels = {_item_label(_resolve_doc_item(it, doc)) for it in doc_items}
    if "table" in labels:
        return "table"
    if labels & {"picture", "figure"}:
        return "figure"
    return "prose"


def build_hybrid_chunker():
    return HybridChunker(repeat_table_header=True, merge_peers=True, always_emit_headings=True)


def chunk_document_to_records(doc, doc_id: str, doc_metadata: dict | None = None,
                               min_chunk_words: int = MIN_CHUNK_WORDS) -> list[dict]:
    chunker = build_hybrid_chunker()
    records, kept = [], 0
    for c in chunker.chunk(doc):
        ctype = get_chunk_type(c, doc)
        raw_text = getattr(c, "text", "") or ""
        embed_text = chunker.contextualize(c) or raw_text
        doc_items = list(getattr(getattr(c, "meta", None), "doc_items", None) or [])
        if ctype == "table":
            markdown_text = _try_table_markdown(doc_items, doc)
            if markdown_text:
                raw_text = embed_text = markdown_text
        if ctype == "prose" and len(raw_text.split()) < min_chunk_words:
            continue
        headings = getattr(getattr(c, "meta", None), "headings", None) or []
        records.append({
            "chunk_id": f"{doc_id}::chunk::{kept}",
            "doc_id": doc_id,
            "chunk_index": kept,
            "chunk_type": ctype,
            "text": raw_text,
            "embed_text": embed_text,
            "token_count": chunker.tokenizer.count_tokens(embed_text),
            "section_path": " > ".join(str(h).strip() for h in headings if str(h).strip()),
            "pages": [],
            "metadata": doc_metadata or {},
        })
        kept += 1
    return records


## Step 3 — two tiny synthetic "papers"

The outcomes-table example from notebooks `01`/`02`, plus a second short
document with no table — enough to later demonstrate resume actually
skipping something on a second run. This step only builds the two
`DoclingDocument`s; chunking them is the next step.

In [ ]:
def make_doc(md_text: str):
    p = Path(tempfile.mkstemp(suffix=".md")[1])
    p.write_text(md_text)
    return DocumentConverter().convert(str(p)).document


doc_a_md = """# Results

Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm. Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm. Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm. Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm. Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm. Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm.

## Outcomes Table

| Patient ID | Treatment Arm | Wound Healing (days) | Complication |
| --- | --- | --- | --- |
| P001 | Early excision | 14 | None |
| P002 | Delayed excision | 21 | Infection |
| P003 | Early excision | 12 | None |
| P004 | Delayed excision | 25 | Graft failure |
| P005 | Early excision | 15 | None |

These results are discussed further in the next section.
"""

doc_b_md = """# Discussion

Early excision was associated with shorter healing times across both cohorts studied here, consistent with prior reports in the burn literature, though the sample size in this synthetic example is too small to draw a real conclusion from.
"""

synthetic_docs = {
    "paper-a": (make_doc(doc_a_md), {"title": "Synthetic outcomes paper", "year": "2026"}),
    "paper-b": (make_doc(doc_b_md), {"title": "Synthetic discussion paper", "year": "2026"}),
}
print(f"built {len(synthetic_docs)} synthetic documents: {list(synthetic_docs.keys())}")


## Step 4 — run the chunker over both synthetic papers

Real output before anything about storage enters the picture: how many
chunks each paper produces.

In [ ]:
for doc_id, (doc, meta) in synthetic_docs.items():
    n = len(chunk_document_to_records(doc, doc_id, meta))
    print(f"{doc_id}: {n} chunks")


## Step 5 — resume bookkeeping

One progress file per destination — a paper already written to Chroma must
not be silently treated as "done" if the target later switches to Pinecone.
Progress files live under a scratch temp directory for this demo, never
under a committed `runs/` path.

In [ ]:
import json
import re

PROGRESS_DIR = Path(tempfile.mkdtemp(prefix="chunk_store_demo_"))


def progress_path(target: str) -> Path:
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", target or "default")
    return PROGRESS_DIR / f"chunk_embed.{safe}.progress.json"


def load_progress(path: Path) -> set:
    if path.exists():
        return set(json.loads(path.read_text()))
    return set()


def save_progress(path: Path, done_ids: set) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(sorted(done_ids)))


## Step 6 — prove the resume check works, before it's wired into any store write

This is the guard the real runs below depend on: an empty progress file
reads back as an empty set, and once a doc id is saved, it reads back as
already done. Confirmed here on its own, with no chunker and no store
involved, before `run()` below ever calls it for real.

In [ ]:
demo_progress_path = progress_path("resume-check-demo")
print(f"before any write: {load_progress(demo_progress_path)}")

save_progress(demo_progress_path, {"paper-a"})
print(f"after saving paper-a: {load_progress(demo_progress_path)}")


## Step 7 — the offline embedding stand-in

`hash_embed` is the same deterministic, non-semantic stand-in as notebook
`01` — this notebook needs no API key to demonstrate a real upsert.

In [ ]:
import hashlib


def hash_embed(texts: list[str], dim: int = 16) -> list[list[float]]:
    """Deterministic, offline, NOT semantically meaningful -- same stand-in
    as notebook 01, so this notebook needs no API key to demonstrate upsert."""
    return [[b / 255.0 for b in hashlib.sha256(t.encode()).digest()[:dim]] for t in texts]


vecs = hash_embed(["hello world", "a second string"])
print(f"{len(vecs)} vectors, dim={len(vecs[0])}")


## Step 8 — open a local Chroma collection

`get_chroma_collection` is file-based: it needs a directory on disk and no
server, no account. Open one here and confirm it starts empty.

In [ ]:
def get_chroma_collection(db_dir: str, collection_name: str):
    path = Path(db_dir)
    path.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(path))
    return client.get_or_create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})


demo_chroma_dir = tempfile.mkdtemp(prefix="chunk_store_demo_chroma_")
demo_col = get_chroma_collection(demo_chroma_dir, "resume_check_demo")
print(f"collection {demo_col.name!r} opened, count={demo_col.count()}")


## Step 9 — upsert into Chroma

`upsert_chroma` writes chunk records plus their embeddings into a collection.
Try it on paper-a's real chunk records and the hash-embed vectors from Step
7, into the throwaway collection opened in Step 8, and confirm the count
moves from 0 to the number of chunks written.

In [ ]:
demo_doc, demo_meta = synthetic_docs["paper-a"]
demo_records = chunk_document_to_records(demo_doc, "paper-a", demo_meta)
demo_embeddings = hash_embed([r["embed_text"] for r in demo_records])


def upsert_chroma(col, records: list[dict], embeddings: list[list[float]]) -> None:
    if not records:
        return
    ids = [r["chunk_id"] for r in records]
    docs = [r["text"][:8000] for r in records]
    metas = []
    for r in records:
        # ChromaDB accepts only str/int/float/bool metadata values.
        clean = {"doc_id": r["doc_id"], "chunk_index": r["chunk_index"], "chunk_type": r["chunk_type"]}
        for k, v in (r.get("metadata") or {}).items():
            clean[k] = v if isinstance(v, (str, int, float, bool)) else str(v)
        metas.append(clean)
    col.upsert(ids=ids, embeddings=embeddings, documents=docs, metadatas=metas)


upsert_chroma(demo_col, demo_records, demo_embeddings)
print(f"collection {demo_col.name!r} after upsert, count={demo_col.count()}")


## Step 10 — the gated backends and the `upsert()` dispatcher

Pinecone and S3 Vectors keep the real client-construction code path, but each
is gated behind a real credential check — calling either without a key
raises a clear, named error, not a stack trace from a missing account deep
in a client library. `upsert()` is the one dispatcher every backend name
routes through.

In [ ]:
import os


def get_pinecone_index_stub(index_name: str = "cookbook-demo"):
    api_key = os.getenv("PINECONE_API_KEY")
    if not api_key:
        raise ValueError(
            "Set PINECONE_API_KEY to use the pinecone backend. No key is "
            "bundled with this cookbook."
        )
    from pinecone import Pinecone  # only imported once a key is actually present
    pc = Pinecone(api_key=api_key)
    return pc.Index(index_name)


def get_s3vectors_stub():
    if not os.getenv("AWS_ACCESS_KEY_ID"):
        raise ValueError(
            "Set AWS credentials to use the s3vectors backend."
        )
    raise NotImplementedError("S3 Vectors client wiring isn't included in this cookbook")


def upsert(backend: str, records: list[dict], embeddings: list[list[float]], **kwargs) -> None:
    """One chunker, store selected by a flag -- not a script per store."""
    if backend == "chroma":
        col = get_chroma_collection(kwargs["chroma_dir"], kwargs["chroma_collection"])
        upsert_chroma(col, records, embeddings)
        print(f"  chroma: upserted {len(records)} vectors -> {kwargs['chroma_collection']} "
              f"({col.count()} total)")
    elif backend == "pinecone":
        index = get_pinecone_index_stub(kwargs.get("index_name", "cookbook-demo"))
        index.upsert(vectors=[{"id": r["chunk_id"], "values": e, "metadata": r["metadata"]}
                               for r, e in zip(records, embeddings)])
    elif backend == "s3vectors":
        get_s3vectors_stub()
    else:
        raise ValueError(f"unknown backend: {backend!r}")


## Step 11 — define `run()`, the flag-selected entry point

Ties the chunker, the resume check (Steps 5-6), and the backend dispatcher
(Step 10) together: `--dry-run` counts chunks with no upserts at all; a real
run skips any doc id already recorded in that destination's progress file,
and saves progress after each successful write.

In [ ]:
def run(backend: str, docs: dict, *, dry_run: bool = False, no_resume: bool = False, **backend_kwargs):
    target_label = backend_kwargs.get("chroma_collection", backend)
    prog_path = progress_path(target_label)
    done_ids = set() if no_resume else load_progress(prog_path)

    all_records = {}
    for doc_id, (doc, meta) in docs.items():
        all_records[doc_id] = chunk_document_to_records(doc, doc_id, meta)

    if dry_run:
        total = sum(len(r) for r in all_records.values())
        print(f"DRY RUN [{backend}]: {total} chunks across {len(docs)} papers, no upserts")
        return

    for doc_id, records in all_records.items():
        if doc_id in done_ids:
            print(f"  skip {doc_id} (already in {target_label!r})")
            continue
        embeddings = hash_embed([r["embed_text"] for r in records])
        upsert(backend, records, embeddings, **backend_kwargs)
        done_ids.add(doc_id)
        save_progress(prog_path, done_ids)
    print(f"progress file: {prog_path}")


## Step 12 — dry-run: count chunks, no upserts, no network

`--dry-run` counts chunks with no API/network calls at all; here that's
simply calling `chunk_document_to_records` without calling `upsert`.

In [ ]:
chroma_dir = tempfile.mkdtemp(prefix="chunk_store_chroma_")
run("chroma", synthetic_docs, dry_run=True, chroma_collection="synthetic_papers", chroma_dir=chroma_dir)


## Step 13 — the real run, against local Chroma

Same call, `dry_run=False`: this actually chunks both papers, embeds them
with the offline stand-in, and upserts into the local Chroma collection.

In [ ]:
run("chroma", synthetic_docs, chroma_collection="synthetic_papers", chroma_dir=chroma_dir)


## Step 14 — run it again: resume should skip both papers

Same backend, same collection, same progress file — both paper ids are
already recorded as done, so this run should skip both and write nothing new.

In [ ]:
run("chroma", synthetic_docs, chroma_collection="synthetic_papers", chroma_dir=chroma_dir)


## Step 15 — the Pinecone-gated path, with no key configured

No `PINECONE_API_KEY` is bundled with this cookbook, so this call is expected
to stop cleanly with a named error — not crash with a raw traceback.

In [ ]:
print("pinecone backend with no key configured")
try:
    run("pinecone", synthetic_docs, dry_run=False, index_name="cookbook-demo")
except ValueError as e:
    print(f"  stopped cleanly: {e}")


## Summary

One chunker (notebook `02`'s `HybridChunker` path), one `upsert()` dispatcher,
three backend names. Only `chroma` is live in this offline cookbook — swap in
a real `PINECONE_API_KEY` / AWS credentials and the same `run()` call reaches
a real index with no code change, which is the point of routing every
backend through one flag-selected function instead of one script per store.